Question 4

In [5]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [6]:
# helper functions
def poly_features(x, p):
    # return [x, x^2, ..., x^p] as an (N, p) matrix
    x = np.asarray(x).reshape(-1)
    return np.column_stack([x**k for k in range(1, p + 1)])

def fit_closed_form(Phi, y):
    # closed-form OLS with intercept using pseudoinverse:
    # w = pinv([1, Phi]) @y
    # w[0] is intercept and w[1:] are coefficients.

    Phi_aug = np.column_stack([np.ones(Phi.shape[0]), Phi])
    w = np.linalg.pinv(Phi_aug) @ y
    return w

def predict_closed_form(Phi, w):
    Phi_aug = np.column_stack([np.ones(Phi.shape[0]), Phi])
    return Phi_aug @ w

In [7]:
# load data
train_path = "train.csv"
test_path  = "test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# target transform: price/1000
y_train = train_df["price"].astype(float).to_numpy() / 1000.0
y_test  = test_df["price"].astype(float).to_numpy() / 1000.0

# single feature: sqft_living
x_train = train_df["sqft_living"].astype(float).to_numpy()
x_test  = test_df["sqft_living"].astype(float).to_numpy()

# train and evaluate for p = 1..5
rows = []
for p in range(1 , 6):
    Phi_train = poly_features(x_train, p)
    Phi_test  = poly_features(x_test, p)

    # scale each polynomial feature column
    scaler = StandardScaler()
    Phi_train_s = scaler.fit_transform(Phi_train)
    Phi_test_s  = scaler.transform(Phi_test)

    # closed-form fit and predict
    w = fit_closed_form(Phi_train_s, y_train)
    pred_train = predict_closed_form(Phi_train_s, w)
    pred_test  = predict_closed_form(Phi_test_s, w)

    # metrics
    train_mse = mean_squared_error(y_train, pred_train)
    test_mse  = mean_squared_error(y_test, pred_test)
    train_r2  = r2_score(y_train, pred_train)
    test_r2   = r2_score(y_test, pred_test)

    rows.append([p, train_mse, train_r2, test_mse, test_r2])

results = pd.DataFrame(rows, columns=["p", "Train MSE", "Train R^2", "Test MSE", "Test R^2"])
print(results.to_string(index = False))

 p    Train MSE  Train R^2      Test MSE  Test R^2
 1 57947.526161   0.496709  88575.978543  0.468736
 2 54822.665116   0.523849  71791.679479  0.569406
 3 53785.194716   0.532860  99833.483763  0.401216
 4 52795.774758   0.541453 250979.274285 -0.505331
 5 52626.111955   0.542927 570616.914821 -2.422464


As the polynomial degree p increases, the training MSE steadily decreases and training R-squared increase, which is expected because a higher degree polynomial is more flexible and can fit the training data better. On the testing set, performance improves from p=1 to p=2, which sugests a small amount of nonlinearity between sqft_living and price. However, for p>=3, test performance gets a lot worse, and by p=4 and p=5, the test R-squared becomes negative.